<a href="https://colab.research.google.com/github/zshaan25/CodeAlpha_Credit_Scoring/blob/main/Credit_Scoring_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, precision_score, recall_score, f1_score

# 1. Fetch Data: Pulls the German Credit Dataset directly into Colab
print("Fetching dataset via OpenML...")
data = fetch_openml(name='credit-g', version=1, as_frame=True, parser='auto')
X = data.data
# Target is 'good' or 'bad' credit risk. We map 'good' to 1, 'bad' to 0.
y = data.target.map({'good': 1, 'bad': 0})

# 2. Preprocessing Setup: Segregate numerical and categorical columns
categorical_cols = X.select_dtypes(include=['category', 'object']).columns
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns

# Build a transformer to scale numbers and encode text categories
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ])

# 3. Model Pipeline: Link preprocessing directly to the Random Forest
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced'))
])

# 4. Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 5. Execute Training
print("Training Random Forest model...")
model_pipeline.fit(X_train, y_train)

# 6. Evaluate Metrics
print("Evaluating model performance...")
predictions = model_pipeline.predict(X_test)
probabilities = model_pipeline.predict_proba(X_test)[:, 1]

print("\n--- Final Model Evaluation ---")
print(f"Precision: {precision_score(y_test, predictions):.4f}")
print(f"Recall:    {recall_score(y_test, predictions):.4f}")
print(f"F1-Score:  {f1_score(y_test, predictions):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, probabilities):.4f}\n")
print("Detailed Classification Report:")
print(classification_report(y_test, predictions))

Fetching dataset via OpenML...
Training Random Forest model...
Evaluating model performance...

--- Final Model Evaluation ---
Precision: 0.8000
Recall:    0.9362
F1-Score:  0.8627
ROC-AUC:   0.8279

Detailed Classification Report:
              precision    recall  f1-score   support

           0       0.74      0.44      0.55        59
           1       0.80      0.94      0.86       141

    accuracy                           0.79       200
   macro avg       0.77      0.69      0.71       200
weighted avg       0.78      0.79      0.77       200

